# Lab 3: Train Image Recognition CNN with Vibe-Coding

## End-to-End CNN Training Using a Vibe-Coding Workflow

In this lab, we will iteratively build and improve a CIFAR-10 image classifier using the **vibe-coding** approach: prompt an AI assistant, generate code, test results, and refine until we reach our target performance.

**Backend:** Keras 3 with PyTorch  
**Dataset:** CIFAR-10 (60,000 32x32 colour images in 10 classes)

In [ ]:
# Run this cell in Google Colab to install dependencies
# Skip if running locally with uv
import sys
if 'google.colab' in sys.modules:
    !pip install -q keras torch torchvision gradio python-dotenv datasets transformers huggingface_hub
    print('Dependencies installed!')

In [ ]:
# Backend setup -- must be set before importing Keras
import os
os.environ["KERAS_BACKEND"] = "torch"

import keras
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

print(f"Keras version: {keras.__version__}")
print(f"Backend: {keras.backend.backend()}")

## Step 1: Vibe-Coding Workflow Overview

The **vibe-coding** workflow is an iterative development cycle powered by AI assistance:

1. **Prompt** -- Describe what you want to build or improve in natural language.
2. **Generate** -- The AI assistant generates code based on your prompt.
3. **Test** -- Run the code, observe metrics (accuracy, loss curves, etc.).
4. **Refine** -- Analyse results, identify weaknesses, and prompt for improvements.

```
Prompt --> Generate --> Test --> Refine
  ^                                |
  |________________________________|
```

Each iteration should produce a measurable improvement. We will go through several iterations in this lab, starting from a bare-bones baseline and ending with a well-tuned model.

In [ ]:
# Load CIFAR-10 dataset
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()

# CIFAR-10 class names
CLASS_NAMES = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
]

# Preprocess: convert to float32, normalise to [0, 1]
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

print(f"Training samples: {x_train.shape[0]}")
print(f"Test samples:     {x_test.shape[0]}")
print(f"Image shape:      {x_train.shape[1:]}")
print(f"Number of classes: {len(CLASS_NAMES)}")

# Visualise a few samples
fig, axes = plt.subplots(1, 8, figsize=(16, 2))
for i, ax in enumerate(axes):
    ax.imshow(x_train[i])
    ax.set_title(CLASS_NAMES[int(y_train[i])])
    ax.axis("off")
plt.suptitle("Sample CIFAR-10 Images")
plt.tight_layout()
plt.show()

## Step 2: Build Baseline Model

Our first iteration is a simple CNN with no augmentation. This gives us a reference accuracy to improve upon.

**Architecture:**
- Two Conv2D + MaxPooling2D blocks
- Flatten + Dense layers
- No data augmentation

In [ ]:
# Baseline model -- simple Sequential CNN
baseline_model = keras.Sequential([
    keras.layers.Input(shape=(32, 32, 3)),

    # Block 1
    keras.layers.Conv2D(32, (3, 3), activation="relu", padding="same"),
    keras.layers.MaxPooling2D((2, 2)),

    # Block 2
    keras.layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
    keras.layers.MaxPooling2D((2, 2)),

    # Classifier head
    keras.layers.Flatten(),
    keras.layers.Dense(64, activation="relu"),
    keras.layers.Dense(10, activation="softmax"),
], name="baseline_cnn")

baseline_model.summary()

# Compile
baseline_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

# Train for 10 epochs
baseline_history = baseline_model.fit(
    x_train, y_train,
    epochs=10,
    batch_size=64,
    validation_data=(x_test, y_test),
)

# Plot training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(baseline_history.history["loss"], label="Train Loss")
ax1.plot(baseline_history.history["val_loss"], label="Val Loss")
ax1.set_title("Baseline -- Loss")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.legend()

ax2.plot(baseline_history.history["accuracy"], label="Train Accuracy")
ax2.plot(baseline_history.history["val_accuracy"], label="Val Accuracy")
ax2.set_title("Baseline -- Accuracy")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy")
ax2.legend()

plt.tight_layout()
plt.show()

baseline_val_acc = max(baseline_history.history["val_accuracy"])
print(f"\nBaseline best validation accuracy: {baseline_val_acc:.4f}")

## Step 3: Add Preprocessing and Augmentation

The baseline likely overfits -- training accuracy is much higher than validation accuracy. Data augmentation artificially increases the diversity of training samples, helping the model generalise.

We add augmentation layers directly into the model so that augmentation is applied on-the-fly during training and automatically disabled during inference.

In [ ]:
# Model with built-in augmentation
augmented_model = keras.Sequential([
    keras.layers.Input(shape=(32, 32, 3)),

    # Preprocessing and augmentation layers
    keras.layers.Rescaling(1.0),  # data already normalised; included to show the pattern
    keras.layers.RandomFlip("horizontal"),
    keras.layers.RandomRotation(0.1),
    keras.layers.RandomZoom(0.1),

    # Block 1
    keras.layers.Conv2D(32, (3, 3), activation="relu", padding="same"),
    keras.layers.MaxPooling2D((2, 2)),

    # Block 2
    keras.layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
    keras.layers.MaxPooling2D((2, 2)),

    # Classifier head
    keras.layers.Flatten(),
    keras.layers.Dense(64, activation="relu"),
    keras.layers.Dense(10, activation="softmax"),
], name="augmented_cnn")

augmented_model.summary()

# Compile
augmented_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

# Train for 10 epochs
augmented_history = augmented_model.fit(
    x_train, y_train,
    epochs=10,
    batch_size=64,
    validation_data=(x_test, y_test),
)

# Plot training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(augmented_history.history["loss"], label="Train Loss")
ax1.plot(augmented_history.history["val_loss"], label="Val Loss")
ax1.set_title("Augmented -- Loss")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.legend()

ax2.plot(augmented_history.history["accuracy"], label="Train Accuracy")
ax2.plot(augmented_history.history["val_accuracy"], label="Val Accuracy")
ax2.set_title("Augmented -- Accuracy")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy")
ax2.legend()

plt.tight_layout()
plt.show()

augmented_val_acc = max(augmented_history.history["val_accuracy"])
print(f"\nAugmented model best validation accuracy: {augmented_val_acc:.4f}")

## Step 4: Vibe-Coding Iteration 1 -- Improve Architecture

Now we apply the vibe-coding cycle: based on the results so far, we ask for architectural improvements.

**Changes in this iteration:**
- Deeper network: three convolutional blocks with increasing filters (32 -> 64 -> 128)
- `BatchNormalization` after each convolutional layer for faster, more stable training
- `GlobalAveragePooling2D` instead of `Flatten` to reduce parameter count and overfitting
- Dropout for additional regularisation

In [ ]:
# Deeper CNN with BatchNormalization and GlobalAveragePooling2D
deeper_model = keras.Sequential([
    keras.layers.Input(shape=(32, 32, 3)),

    # Augmentation
    keras.layers.RandomFlip("horizontal"),
    keras.layers.RandomRotation(0.1),
    keras.layers.RandomZoom(0.1),

    # Block 1: 32 filters
    keras.layers.Conv2D(32, (3, 3), padding="same"),
    keras.layers.BatchNormalization(),
    keras.layers.Activation("relu"),
    keras.layers.Conv2D(32, (3, 3), padding="same"),
    keras.layers.BatchNormalization(),
    keras.layers.Activation("relu"),
    keras.layers.MaxPooling2D((2, 2)),
    keras.layers.Dropout(0.25),

    # Block 2: 64 filters
    keras.layers.Conv2D(64, (3, 3), padding="same"),
    keras.layers.BatchNormalization(),
    keras.layers.Activation("relu"),
    keras.layers.Conv2D(64, (3, 3), padding="same"),
    keras.layers.BatchNormalization(),
    keras.layers.Activation("relu"),
    keras.layers.MaxPooling2D((2, 2)),
    keras.layers.Dropout(0.25),

    # Block 3: 128 filters
    keras.layers.Conv2D(128, (3, 3), padding="same"),
    keras.layers.BatchNormalization(),
    keras.layers.Activation("relu"),
    keras.layers.Conv2D(128, (3, 3), padding="same"),
    keras.layers.BatchNormalization(),
    keras.layers.Activation("relu"),

    # Global average pooling instead of Flatten
    keras.layers.GlobalAveragePooling2D(),
    keras.layers.Dropout(0.5),

    # Classifier head
    keras.layers.Dense(128, activation="relu"),
    keras.layers.Dense(10, activation="softmax"),
], name="deeper_cnn")

deeper_model.summary()

# Compile
deeper_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

# Train for 10 epochs
deeper_history = deeper_model.fit(
    x_train, y_train,
    epochs=10,
    batch_size=64,
    validation_data=(x_test, y_test),
)

# Plot training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(deeper_history.history["loss"], label="Train Loss")
ax1.plot(deeper_history.history["val_loss"], label="Val Loss")
ax1.set_title("Deeper CNN -- Loss")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.legend()

ax2.plot(deeper_history.history["accuracy"], label="Train Accuracy")
ax2.plot(deeper_history.history["val_accuracy"], label="Val Accuracy")
ax2.set_title("Deeper CNN -- Accuracy")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy")
ax2.legend()

plt.tight_layout()
plt.show()

deeper_val_acc = max(deeper_history.history["val_accuracy"])
print(f"\nDeeper CNN best validation accuracy: {deeper_val_acc:.4f}")

## Step 5: Vibe-Coding Iteration 2 -- Tune Hyperparameters

The architecture is now solid. In this iteration we focus on training strategy:

- **EarlyStopping** (patience=5): stop training when validation loss stops improving.
- **ReduceLROnPlateau**: reduce the learning rate when the model plateaus.
- **ModelCheckpoint**: save the best model weights automatically.
- Train for up to 30 epochs -- early stopping will halt training if needed.

In [ ]:
# Rebuild the deeper model (fresh weights)
tuned_model = keras.Sequential([
    keras.layers.Input(shape=(32, 32, 3)),

    # Augmentation
    keras.layers.RandomFlip("horizontal"),
    keras.layers.RandomRotation(0.1),
    keras.layers.RandomZoom(0.1),

    # Block 1: 32 filters
    keras.layers.Conv2D(32, (3, 3), padding="same"),
    keras.layers.BatchNormalization(),
    keras.layers.Activation("relu"),
    keras.layers.Conv2D(32, (3, 3), padding="same"),
    keras.layers.BatchNormalization(),
    keras.layers.Activation("relu"),
    keras.layers.MaxPooling2D((2, 2)),
    keras.layers.Dropout(0.25),

    # Block 2: 64 filters
    keras.layers.Conv2D(64, (3, 3), padding="same"),
    keras.layers.BatchNormalization(),
    keras.layers.Activation("relu"),
    keras.layers.Conv2D(64, (3, 3), padding="same"),
    keras.layers.BatchNormalization(),
    keras.layers.Activation("relu"),
    keras.layers.MaxPooling2D((2, 2)),
    keras.layers.Dropout(0.25),

    # Block 3: 128 filters
    keras.layers.Conv2D(128, (3, 3), padding="same"),
    keras.layers.BatchNormalization(),
    keras.layers.Activation("relu"),
    keras.layers.Conv2D(128, (3, 3), padding="same"),
    keras.layers.BatchNormalization(),
    keras.layers.Activation("relu"),

    # Global average pooling
    keras.layers.GlobalAveragePooling2D(),
    keras.layers.Dropout(0.5),

    # Classifier head
    keras.layers.Dense(128, activation="relu"),
    keras.layers.Dense(10, activation="softmax"),
], name="tuned_cnn")

# Compile
tuned_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

# Callbacks
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
        verbose=1,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1,
    ),
    keras.callbacks.ModelCheckpoint(
        filepath="best_cifar10_model.keras",
        monitor="val_accuracy",
        save_best_only=True,
        verbose=1,
    ),
]

# Train for up to 30 epochs
tuned_history = tuned_model.fit(
    x_train, y_train,
    epochs=30,
    batch_size=64,
    validation_data=(x_test, y_test),
    callbacks=callbacks,
)

# Plot training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(tuned_history.history["loss"], label="Train Loss")
ax1.plot(tuned_history.history["val_loss"], label="Val Loss")
ax1.set_title("Tuned CNN -- Loss")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.legend()

ax2.plot(tuned_history.history["accuracy"], label="Train Accuracy")
ax2.plot(tuned_history.history["val_accuracy"], label="Val Accuracy")
ax2.set_title("Tuned CNN -- Accuracy")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy")
ax2.legend()

plt.tight_layout()
plt.show()

tuned_val_acc = max(tuned_history.history["val_accuracy"])
print(f"\nTuned CNN best validation accuracy: {tuned_val_acc:.4f}")

## Step 6: Compare All Iterations

Let us bring all results together to see how each iteration improved performance.

In [ ]:
# Comparison table
comparison_df = pd.DataFrame({
    "Iteration": ["Baseline", "+ Augmentation", "+ Deeper Architecture", "+ Hyperparameter Tuning"],
    "Best Val Accuracy": [
        baseline_val_acc,
        augmented_val_acc,
        deeper_val_acc,
        tuned_val_acc,
    ],
})
comparison_df["Best Val Accuracy"] = comparison_df["Best Val Accuracy"].map("{:.4f}".format)
print(comparison_df.to_string(index=False))

# Plot all validation accuracy curves together
plt.figure(figsize=(10, 5))
plt.plot(baseline_history.history["val_accuracy"], label="Baseline", linestyle="--")
plt.plot(augmented_history.history["val_accuracy"], label="+ Augmentation", linestyle="--")
plt.plot(deeper_history.history["val_accuracy"], label="+ Deeper Architecture")
plt.plot(tuned_history.history["val_accuracy"], label="+ Hyperparameter Tuning")
plt.title("Validation Accuracy Across All Iterations")
plt.xlabel("Epoch")
plt.ylabel("Validation Accuracy")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Step 7: Save the Best Model

We save the final tuned model in Keras `.keras` format and verify that it loads correctly.

In [ ]:
# Save the best model
tuned_model.save("best_cifar10_model.keras")
print("Model saved to best_cifar10_model.keras")

# Verify loading
loaded_model = keras.saving.load_model("best_cifar10_model.keras")
print("Model loaded successfully.")

# Verify predictions match
sample_images = x_test[:5]
original_preds = tuned_model.predict(sample_images)
loaded_preds = loaded_model.predict(sample_images)

print("\nPrediction comparison (original vs loaded):")
for i in range(5):
    orig_class = CLASS_NAMES[np.argmax(original_preds[i])]
    load_class = CLASS_NAMES[np.argmax(loaded_preds[i])]
    true_class = CLASS_NAMES[int(y_test[i])]
    match = "OK" if orig_class == load_class else "MISMATCH"
    print(f"  Image {i}: True={true_class}, Original={orig_class}, Loaded={load_class} [{match}]")

print("\nLab 3 complete. The best model is saved and ready for use in Lab 4.")